# 07: PersistentVolume, ConfigMap & Secrets
Hands-on lab: PostgreSQL with persistent storage, ConfigMaps, Secrets, and rolling updates.

## Cluster Setup
```bash
kind create cluster --name k8-lab
kubectl create namespace dev
```

## 1. Frontend (static React app)
```bash
cd 07.pv,configmap,secrers/frontend
docker build -t frontend-07:1.0 .
kind load docker-image frontend-07:1.0 --name k8-lab
kubectl apply -f frontend.yaml
```
- 3 Pods, ClusterIP service

## 2. Backend v1.0 — Fake JSON
```bash
cd 07.pv,configmap,secrers/backend
docker build -t backend-07:1.0 .
kind load docker-image backend-07:1.0 --name k8-lab
kubectl apply -f backend.yaml
kubectl port-forward service/backend 8080:80 -n dev
```
Returned hardcoded `{"users":[{"id":1,"name":"Alice"},...]}`

## 3. PostgreSQL — Persistent Storage

Three new resource types introduced here: Secret, PVC, StatefulSet.

### Secret (`secrets.yaml`)

Stores sensitive data. Base64-encoded internally, but **not encrypted** — anyone can decode it. Never commit real secrets to Git in production.

```yaml
# Secret — stores sensitive data (base64-encoded internally)
# Unlike ConfigMap, values are NOT stored in plain text.
# But base64 is encoding, NOT encryption — anyone can decode.
apiVersion: v1
kind: Secret

metadata:
  name: postgresql-secret   # referenced by StatefulSet via secretKeyRef
  namespace: dev

type: Opaque                # arbitrary key-value pairs

# stringData accepts plain text; K8s base64-encodes it on store
stringData:
  POSTGRES_USER: admin
  POSTGRES_PASSWORD: password123
```
```bash
kubectl apply -f secrets.yaml
```

### PVC (`pvc.yaml`)

A request for storage. K8s automatically binds it to a matching PersistentVolume. The Pod never sees the PV directly — PVC decouples storage provisioning from Pod definitions.

```yaml
# PersistentVolumeClaim — Pod requests storage via this
# The Pod never sees the PV directly — PVC decouples them.
apiVersion: v1
kind: PersistentVolumeClaim

metadata:
  name: postgres-pvc
  namespace: dev

spec:
  accessModes:
    - ReadWriteOnce      # single node can mount as read-write
  resources:
    requests:
      storage: 1Gi       # minimum capacity requested
```
```bash
kubectl apply -f pvc.yaml
```

### StatefulSet (`statefulset.yaml`)

Like Deployment but for stateful apps. Stable Pod identity (`postgres-0`), ordered creation/termination, and automatic per-Pod PVC via `volumeClaimTemplates`.

```yaml
# StatefulSet — for stateful apps (databases)
# vs Deployment: stable Pod names (postgres-0), ordered startup,
# and each replica gets its own PVC automatically.
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: postgres
  namespace: dev
spec:
  serviceName: postgres           # must match headless service name
  replicas: 1
  selector:
    matchLabels:
      app: postgres
  template:
    metadata:
      labels:
        app: postgres
    spec:
      containers:
      - name: postgres
        image: postgres:16
        ports:
        - containerPort: 5432
        env:
        # Plain value — from ConfigMap in production
        - name: POSTGRES_DB
          value: appdb

        # Sensitive values from Secret — never hardcoded
        - name: POSTGRES_USER
          valueFrom:
            secretKeyRef:
              name: postgresql-secret
              key: POSTGRES_USER
        - name: POSTGRES_PASSWORD
          valueFrom:
            secretKeyRef:
              name: postgresql-secret
              key: POSTGRES_PASSWORD

        # Mount PVC to Postgres data directory
        volumeMounts:
        - name: postgres-storage
          mountPath: /var/lib/postgresql/data

  # volumeClaimTemplates — creates PVC per replica
  # For 1 replica → one PVC named: postgres-storage-postgres-0
  volumeClaimTemplates:
  - metadata:
      name: postgres-storage
    spec:
      accessModes: [ "ReadWriteOnce" ]
      resources:
        requests:
          storage: 2Gi
```

> **Gotcha**: Initially wrote `volumes:` under `spec.template.spec.containers[0]`. Error: `unknown field "volumes"`. In StatefulSet, persistent storage must be declared via `volumeClaimTemplates` at `spec` level, not inside the container spec.

### Service (`service.yaml`)

A headless ClusterIP service (`clusterIP: None`) for stable DNS lookups: `postgres-0.postgres.dev.svc.cluster.local`

```bash
kubectl apply -f statefulset.yaml
kubectl apply -f service.yaml
```

## 4. Backend v1.1 — Real Postgres Connection

Updated `main.py` to use environment variables:
```python
from fastapi import FastAPI
import psycopg2
import os

app = FastAPI()

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("DB_PORT", "5432")
)

@app.get("/")
def get_users():
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users")
    rows = cursor.fetchall()
    cursor.close()
    return {"users": rows}
```

```bash
docker build -t backend-07:1.1 .
kind load docker-image backend-07:1.1 --name k8-lab
kubectl apply -f backend.yaml
kubectl rollout status deployment/backend-deployment -n dev
```

## 5. Rolling Update

Changed container image tag in `backend.yaml` from `backend-07:1.0` → `backend-07:1.1`.

```
Old Pod
    ↓
New Pod (1.1)
    ↓
Old Pod deleted
    ↓
Repeat until all Pods updated
```

## Architecture Diagram
```
Browser
   │
   ▼
frontend-service (ClusterIP)
   │
   ▼
Frontend Pods (3)
   │  HTTP GET /users
   ▼
backend-service (ClusterIP)
   │
   ▼
FastAPI Pods (3)
   │  Reads ConfigMap & Secret (env vars)
   │  psycopg2 connection
   ▼
postgres-service (ClusterIP)
   │
   ▼
PostgreSQL StatefulSet
   │
   ▼
PVC
   │
   ▼
PV (kind host)
```

## Key Takeaways
- **ConfigMap**: non-sensitive config → `os.getenv()`
- **Secret**: sensitive data → `os.getenv()` via `secretKeyRef`
- **No hardcoded values** in application code — all config injected by K8s
- **StatefulSet** ≠ Deployment: stable Pod identity (`postgres-0`), ordered creation, uses `volumeClaimTemplates`
- **PVC** survives Pod restarts — data persists
- **Rolling update** = zero-downtime by changing image tag